# Social History data analysis (Snowflake, read-only)

One row = one social history factor. **Read-only**: `SELECT` / `DESCRIBE` only.

**Grain:** `SocialHistoryId` should be unique. One patient can have many rows (alcohol, housing, pets, ...).

**Value is the main field.** The dictionary lists 10 categories with example phrases. Vendors often store the **example sentence** (`Drinks alcohol`) or a custom string, not the category title. We always list **unique stored Values with occurrences**. A second cell is a **best-effort keyword map** onto the 10 buckets; unmapped values stay visible.

There is **no SNOMED** on this table.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "SOCIAL_HISTORY"   # try SOCIALHISTORY, SOCIAL_HX, SDH

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "social_history_id": "SocialHistoryId",
    "encounter_id": "EncounterId/VisitId",
    "patient_id": "Member/PatientId",
    "value": "Value",
    "date": "Date",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_ID = col("social_history_id")
C_ENC = col("encounter_id")
C_PT = col("patient_id")
C_VAL = col("value")
C_DATE = col("date")

for name, value in [
    ("DB", DB),
    ("T", T),
    ("C_ID", C_ID),
    ("C_ENC", C_ENC),
    ("C_PT", C_PT),
    ("C_VAL", C_VAL),
    ("C_DATE", C_DATE),
]:
    print(f"{name} = {value}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%SOCIAL%'
     OR UPPER(TABLE_NAME) LIKE '%SDH%'
     OR UPPER(TABLE_NAME) LIKE '%SDOH%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT *
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_ID}}) AS UNIQUE_SOCIAL_HISTORY_IDS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_VALUES,
    COUNT(*) - COUNT(DISTINCT {{C_ID}}) AS EXTRA_ROWS_VS_UNIQUE_ID,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_ENTRIES_PER_PATIENT
FROM {{T}};

## 6. Completeness (nulls)

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_ID}} IS NULL, 1, 0)) AS NULL_SOCIAL_HISTORY_ID,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_VAL}} IS NULL, 1, 0)) AS NULL_VALUE,
    SUM(IFF({{C_VAL}} IS NOT NULL AND TRIM({{C_VAL}}::STRING) = '', 1, 0)) AS BLANK_VALUE,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS NULL_DATE
FROM {{T}};

## 7. Value — unique factors with occurrences (the list you need)

One row per stored string. `ROW_COUNT` is how often that exact text appears.

Dictionary examples (not hardcoded as the only allowed values):
1. Substance use — Drinks alcohol, Uses tobacco, Uses vape products
2. Sexual and relationship history
3. Social and living conditions
4. Education and occupation
5. Hobbies and activities
6. Environmental and exposure history
7. Pet ownership
8. Travel history
9. Physical and health-related factors
10. Technology and screen time usage

In [ ]:
SELECT
    {{C_VAL}} AS SOCIAL_HISTORY_VALUE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC, SOCIAL_HISTORY_VALUE;

In [ ]:
WITH tokens AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        TRIM(f.VALUE::STRING) AS WORD
    FROM {{T}},
         LATERAL FLATTEN(
             INPUT => SPLIT(
                 TRIM(REGEXP_REPLACE({{C_VAL}}::STRING, '[^A-Za-z0-9]+', ' ')),
                 ' '
             )
         ) f
    WHERE {{C_VAL}} IS NOT NULL
)
SELECT
    WORD,
    COUNT(*) AS WORD_OCCURRENCES,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_WORD_OCCURRENCES
FROM tokens
WHERE WORD IS NOT NULL
  AND WORD <> ''
GROUP BY 1
ORDER BY WORD_OCCURRENCES DESC, WORD;

## 8. Map Value onto the 10 dictionary categories (best-effort)

`value_mapped_to_bucket` uses keywords. It can mis-tag (for example `smoke exposure` is environmental, but `smoke` might also hit substance). **`value_counts` is the source of truth.** `unmapped_values` is the cleanup list.

In [ ]:
WITH mapped AS (
    SELECT
        {{C_VAL}} AS SOCIAL_HISTORY_VALUE,
        {{C_PT}} AS PATIENT_ID,
        CASE
            WHEN {{C_VAL}} IS NULL THEN 'Unknown (missing Value)'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'alcohol|tobacco|vape|cigar|cannabis|marijuana|substance use', 'i') THEN '1. Substance use'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'sexually|sexual|partner|same[- ]sex|relationship', 'i') THEN '2. Sexual and relationship history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'lives alone|housing|homeless|afford medical|food insecur|living condition', 'i') THEN '3. Social and living conditions'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'school|ged|bachelor|degree|certification|occupat|employ|job|unemploy', 'i') THEN '4. Education and occupation'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'hobb|sport|music|activit', 'i') THEN '5. Hobbies and activities'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'chemical|mold|water damage|smoke exposure|secondhand|environment', 'i') THEN '6. Environmental and exposure history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'dog|cat|pets|pet ownership', 'i') THEN '7. Pet ownership'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'travel', 'i') THEN '8. Travel history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'weight|stress|transfusion|physical and health', 'i') THEN '9. Physical and health-related factors'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'phone|computer|screen|technology', 'i') THEN '10. Technology and screen time usage'
            ELSE 'Unmapped - see Value as stored'
        END AS VALUE_BUCKET
    FROM {{T}}
)
SELECT
    VALUE_BUCKET,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT SOCIAL_HISTORY_VALUE) AS DISTINCT_STORED_VALUES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM mapped
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
WITH mapped AS (
    SELECT
        {{C_VAL}} AS SOCIAL_HISTORY_VALUE,
        {{C_PT}} AS PATIENT_ID,
        CASE
            WHEN {{C_VAL}} IS NULL THEN 'Unknown (missing Value)'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'alcohol|tobacco|vape|cigar|cannabis|marijuana|substance use', 'i') THEN '1. Substance use'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'sexually|sexual|partner|same[- ]sex|relationship', 'i') THEN '2. Sexual and relationship history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'lives alone|housing|homeless|afford medical|food insecur|living condition', 'i') THEN '3. Social and living conditions'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'school|ged|bachelor|degree|certification|occupat|employ|job|unemploy', 'i') THEN '4. Education and occupation'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'hobb|sport|music|activit', 'i') THEN '5. Hobbies and activities'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'chemical|mold|water damage|smoke exposure|secondhand|environment', 'i') THEN '6. Environmental and exposure history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'dog|cat|pets|pet ownership', 'i') THEN '7. Pet ownership'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'travel', 'i') THEN '8. Travel history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'weight|stress|transfusion|physical and health', 'i') THEN '9. Physical and health-related factors'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'phone|computer|screen|technology', 'i') THEN '10. Technology and screen time usage'
            ELSE 'Unmapped - see Value as stored'
        END AS VALUE_BUCKET
    FROM {{T}}
)
SELECT
    VALUE_BUCKET,
    SOCIAL_HISTORY_VALUE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM mapped
GROUP BY 1, 2
ORDER BY VALUE_BUCKET, ROW_COUNT DESC;

In [ ]:
WITH mapped AS (
    SELECT
        {{C_VAL}} AS SOCIAL_HISTORY_VALUE,
        CASE
            WHEN {{C_VAL}} IS NULL THEN 'Unknown (missing Value)'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'alcohol|tobacco|vape|cigar|cannabis|marijuana|substance use', 'i') THEN '1. Substance use'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'sexually|sexual|partner|same[- ]sex|relationship', 'i') THEN '2. Sexual and relationship history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'lives alone|housing|homeless|afford medical|food insecur|living condition', 'i') THEN '3. Social and living conditions'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'school|ged|bachelor|degree|certification|occupat|employ|job|unemploy', 'i') THEN '4. Education and occupation'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'hobb|sport|music|activit', 'i') THEN '5. Hobbies and activities'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'chemical|mold|water damage|smoke exposure|secondhand|environment', 'i') THEN '6. Environmental and exposure history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'dog|cat|pets|pet ownership', 'i') THEN '7. Pet ownership'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'travel', 'i') THEN '8. Travel history'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'weight|stress|transfusion|physical and health', 'i') THEN '9. Physical and health-related factors'
            WHEN REGEXP_LIKE({{C_VAL}}::STRING, 'phone|computer|screen|technology', 'i') THEN '10. Technology and screen time usage'
            ELSE 'Unmapped - see Value as stored'
        END AS VALUE_BUCKET
    FROM {{T}}
)
SELECT
    SOCIAL_HISTORY_VALUE,
    COUNT(*) AS ROW_COUNT,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM mapped
WHERE VALUE_BUCKET = 'Unmapped - see Value as stored'
GROUP BY 1
ORDER BY ROW_COUNT DESC;

## 9. Date distribution

`Date` is when the social history was recorded.

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_DATE}}) AS MIN_DATE,
    MAX({{C_DATE}}) AS MAX_DATE,
    SUM(IFF({{C_DATE}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_ROWS,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS MISSING_DATE_ROWS
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_DATE}}) AS RECORD_YEAR,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_VALUES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_DATE}} IS NULL THEN 90
            WHEN {{C_DATE}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 10. Entries per patient and one patient drill-down

In [ ]:
WITH per_pt AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        COUNT(*) AS ROW_COUNT
    FROM {{T}}
    WHERE {{C_PT}} IS NOT NULL
    GROUP BY 1
)
SELECT
    ROW_COUNT AS ENTRIES_PER_PATIENT,
    COUNT(*) AS NUMBER_OF_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_PATIENTS
FROM per_pt
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_VAL}}) AS UNIQUE_VALUES,
    MIN({{C_DATE}}) AS FIRST_DATE,
    MAX({{C_DATE}}) AS LAST_DATE
FROM {{T}}
WHERE {{C_PT}} IS NOT NULL
GROUP BY 1
ORDER BY ROW_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_ID}} AS SOCIAL_HISTORY_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_VAL}} AS SOCIAL_HISTORY_VALUE,
    {{C_DATE}} AS RECORD_DATE
FROM {{T}}
WHERE {{C_PT}} = (
        SELECT {{C_PT}}
        FROM {{T}}
        WHERE {{C_PT}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_DATE}} NULLS LAST, {{C_ID}};

## Notes

- **Downloading:** use the download arrow on each SQL result grid.
- Trust `value_counts` over the keyword buckets if they disagree.
- Snowflake POSIX regex may not support `\\b` word boundaries; if pet mapping looks wrong, still use the unique Value list.
- Run `config` before SQL cells.